# 11 · Stock clips for the test plan (rows 1–24) + fit check

Downloads free Pexels clips (Pexels license: free to use, no attribution required; sources are listed in
`videos/rows/_sources.csv` anyway), makes two edited clips (row 13: legs out of frame via a virtual camera;
row 22: 480p re-encode), then checks every clip: people per frame, how much of the frame the person fills,
whether the box touches the border, brightness. Contact sheets go to `videos/rows/_preview/`.

Rows 7 and 9 are the team's own clips (`clipB`, `clipA`); rows 15 and 19 have no stock clip yet; rows 25–29
are Blender checks. After this notebook, re-run notebook 09: it only processes clips it hasn't seen.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/CS747_project"
CODE_DIR    = f"{PROJECT_DIR}/code/rw2b"          # the rw2b folder (pose_lib/, blender/, notebooks/)

from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, time, gc, shutil, subprocess
import numpy as np
if not os.path.exists(f"{CODE_DIR}/pose_lib"):              # first run: unpack the code next to the zip
    !cd "{PROJECT_DIR}/code" && unzip -oq rw2b.zip
sys.path.insert(0, CODE_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics
import pandas as pd
from pose_lib import clips as C
from ultralytics import YOLO
yolo = YOLO("yolo11n.pt")
ROWS = f"{PROJECT_DIR}/videos/rows"; PREV = f"{ROWS}/_preview"
os.makedirs(PREV, exist_ok=True)

## 1. Download

In [ ]:
src, failed = [], []
jobs = [(r, pid, d, 1) for r, (pid, d) in C.PEXELS.items()] + \
       [(r, pid, d, 2 + i) for r, ex in C.EXTRA.items() for i, (pid, d) in enumerate(ex)]
for row, pid, desc, k in jobs:
    dst = f"{ROWS}/{C.name(row, desc, k)}"
    try:
        C.download(pid, dst); src.append({"file": os.path.basename(dst), "row": row, "pexels_id": pid,
                                          "url": f"https://www.pexels.com/video/{pid}/"})
        print("ok  ", os.path.basename(dst), f"{os.path.getsize(dst)/1e6:.1f} MB")
    except Exception as e:
        failed.append((row, pid, str(e)[:120])); print("FAIL", row, pid, e)
pd.DataFrame(src).to_csv(f"{ROWS}/_sources.csv", index=False)
print(len(src), "downloaded,", len(failed), "failed")

## 2. Edited clips (rows 13 and 22)

In [ ]:
r16 = f"{ROWS}/{C.name(16, 'tripod')}"; r1 = f"{ROWS}/{C.name(1, 'slowwalk')}"
e13 = f"{ROWS}/{C.name(13, 'upperbody', edit=True)}"; e22 = f"{ROWS}/{C.name(22, 'lowres', edit=True)}"
if os.path.exists(r16) and not os.path.exists(e13): C.upper_body_cam(r16, e13, yolo)
if os.path.exists(r1) and not os.path.exists(e22): C.to_480p(r1, e22)

## 3. Fit check + contact sheets

In [ ]:
rows = []
for p in sorted(glob.glob(f"{ROWS}/row*.mp4")):
    n = os.path.basename(p)[:-4]
    r = C.check(p, yolo, sheet=f"{PREV}/{n}.jpg"); r["clip"] = n; rows.append(r)
chk = pd.DataFrame(rows).set_index("clip"); chk.to_csv(f"{ROWS}/_check.csv"); chk